In [ ]:
#| default_exp solve_auth

# SolveIt Auth

> Google Sign-In for apps hosted on SolveIt

This module wraps the SolveIt auth service so your FastHTML app can add Google Sign-In with just two function calls: `setup_solve_signin` to add the sign-in route, and `sub_from_signin` to validate the reply and get the user's Google sub ID.

## Setup -

In [ ]:
#| export
from starlette.responses import RedirectResponse

import os, json, httpx, jwt

In [ ]:
#| export
class SolveSigninError(Exception): pass

_AUTH_URL = 'https://auth.solve.it.com/request_signin'

def _key(): return os.environ['AAI_USER_KEY']
def _app_url(port): return f'https://{json.loads(os.environ["PUBLIC_DOMAINS"])[str(port)]}.solve.it.com'

`SolveSigninError` is raised when authentication fails — either from an invalid JWT or an error returned by the auth service. Catch this in your `signin_completed` route to handle auth failures gracefully.

In [ ]:
#| export
solve_signin_rt = '/solve_signin'
def setup_solve_signin(app, port=8000, callback_rt='/signin_completed'):
    "Add /solve_signin route to `app` that redirects to Google via SolveIt auth"
    callback = _app_url(port) + callback_rt
    @app.route(solve_signin_rt)
    def solve_signin():
        r = httpx.post(_AUTH_URL, headers={'Authorization': _key()}, json={'callback_url': callback})
        return RedirectResponse(r.json()['signin_url'])

`setup_solve_signin` adds a `/solve_signin` route to your FastHTML app. When a user visits this route, they're redirected to Google Sign-In via the SolveIt auth service. After authentication, Google redirects back to your `callback_rt` (defaults to `/signin_completed`) with a `signin_reply` query parameter.

The `solve_signin_rt` variable is exported so you can reference it in links: `A('Sign In', href=solve_signin_rt)`.

In [ ]:
#| export
def sub_from_signin(session, signin_reply):
    "Decode signin JWT, validate, and return Google sub ID. Raises SolveSigninError on failure."
    try:
        payload = jwt.decode(signin_reply, _key(), algorithms=['HS256'])
        if 'err' in payload: raise SolveSigninError(payload['err'])
        return str(payload['sub'])
    except jwt.PyJWTError as e: raise SolveSigninError(f'JWT validation failed: {e}') from e

`sub_from_signin` is the function your app calls in the signin completion route. It decodes the JWT reply from SolveIt's auth service, checks for errors, and returns the user's Google sub ID as a string. Raises `SolveSigninError` on any failure.

## Export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()